# Notebook 11 - H1 - N-Gram Analysis

Within this Jupyter Notebook, the aim is to find meaningful two and three word phrases in the H1 corpus that carry more meaning than single words alone. 

Chai argues that computing n-grams outputs clearer and better results when it is done after stopword removal and lemmatization.
Additionally, setting a minimum frequency cutoff (she uses 5 as a cutoff) lets one define an n-gram as an actual n-gram, meaning it appears frequently enough in the corpus and thus is meaningful. (Chai, 2023, 529, 530)

### Sources
- Christine P. Chai. Comparison of text preprocessing methods. Natural Language Engineering, 29(3):509–553, 2023. doi: 10.1017/S1351324922000213.
- N-Gram Language Modelling with NLTK. (14:33:39+00:00). GeeksforGeeks. https://www.geeksforgeeks.org/nlp/n-gram-language-modelling-with-nltk/
- Python Dictionary Comprehension. (09:37:43+00:00). GeeksforGeeks. https://www.geeksforgeeks.org/python/python-dictionary-comprehension/


### 1. Imports

I am using [this documentation](https://www.geeksforgeeks.org/nlp/n-gram-language-modelling-with-nltk/) for support.

In [22]:
import pandas as pd
import ast
from nltk.util import ngrams
from nltk.probability import FreqDist

### 2. Loading the Dataset

In [23]:
df_h1 = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/09_h1_lemmatized.csv")
df_h1["tokens"] = df_h1["tokens"].apply(ast.literal_eval)

### 3. Combining all tokens into one list

In [24]:
all_tokens = [token for tokens in df_h1["tokens"] for token in tokens]
print(f"Total tokens: {len(all_tokens)}")

Total tokens: 80180


### 4. Generating bigrams and trigrams
Using [this](https://www.geeksforgeeks.org/nlp/n-gram-language-modelling-with-nltk/) for help. 

In [25]:
# bigrams = two word phrases
bigrams  = list(ngrams(all_tokens, 2))
# trigrams = three word phrases
trigrams = list(ngrams(all_tokens, 3))

print(f"Total bigrams:  {len(bigrams)}")
print(f"Total trigrams: {len(trigrams)}")

Total bigrams:  80179
Total trigrams: 80178


### 5. Counting the Frequencies

In [26]:
bigram_freq  = FreqDist(bigrams)
trigram_freq = FreqDist(trigrams)

# looking at the 20 most common ones
print("Top 20 bigrams:")
print(bigram_freq.most_common(20))
print("---")
print("Top 20 trigrams:")
print(trigram_freq.most_common(20))

Top 20 bigrams:
[(('black', 'woman'), 74), (('new', 'york'), 73), (('feminist', 'art'), 44), (('would', 'like'), 34), (('art', 'politics'), 32), (('woman', 'work'), 30), (('year', 'ago'), 28), (('woman', 'not'), 28), (('year', 'old'), 26), (('work', 'together'), 22), (('no', 'one'), 21), (('woman', 'art'), 21), (('woman', 'movement'), 20), (('woman', 'music'), 20), (('one', 'day'), 20), (('many', 'woman'), 20), (('woman', 'aware'), 19), (('woman', 'make'), 19), (('woman', 'woman'), 19), (('first', 'time'), 19)]
---
Top 20 trigrams:
[(('feminist', 'art', 'politics'), 19), (('woman', 'aware', 'historically'), 18), (('art', 'politics', 'publish'), 16), (('devote', 'examination', 'art'), 15), (('aware', 'historically', 'connection'), 15), (('new', 'york', 'city'), 12), (('queed', 'ob', 'spade'), 11), (('politics', 'publish', 'winter'), 10), (('structure', 'collective', 'feminist'), 9), (('live', 'new', 'york'), 9), (('guideline', 'contributor', 'issue'), 9), (('third', 'world', 'woman'), 9

### 6. Applying a minimum frequency cutoff

Using [this documentation for support.](https://www.geeksforgeeks.org/python/python-dictionary-comprehension/)

In [27]:
# following Chai's recommendation (only keep ngrams that appear at least 5 times)
min_frequency = 5

filtered_bigrams  = {gram: freq for gram, freq in bigram_freq.items() if freq >= min_frequency}
filtered_trigrams = {gram: freq for gram, freq in trigram_freq.items() if freq >= min_frequency}

print(f"Bigrams  after cutoff: {len(filtered_bigrams)}")
print(f"Trigrams after cutoff: {len(filtered_trigrams)}")

Bigrams  after cutoff: 393
Trigrams after cutoff: 42


#### Filtering out certain Bigrams and Trigrams. 
Certain ngrams are too generic. For example there is the bigram "new, york" and the trigram "feminist, art, politics".
It does not make sense to track it, because in every issue, there is "A feminist publication on art and politics". this does not represent a useful trigram. 
So, before saving the dataframe, I would like to filter out some ngrams. 

In [28]:
df_bigrams = pd.DataFrame(
    filtered_bigrams.items(),
    columns=["bigram", "frequency"]
).sort_values("frequency", ascending=False)

df_trigrams = pd.DataFrame(
    filtered_trigrams.items(),
    columns=["trigram", "frequency"]
).sort_values("frequency", ascending=False)


In [29]:
print(df_bigrams.head(10))
print(df_trigrams.head(10))

               bigram  frequency
32     (black, woman)         74
57        (new, york)         73
9     (feminist, art)         44
21      (would, like)         34
2     (art, politics)         32
59      (woman, work)         30
54       (woman, not)         28
33        (year, ago)         28
256       (year, old)         26
229  (work, together)         22
                              trigram  frequency
6           (feminist, art, politics)         19
4        (woman, aware, historically)         18
7            (art, politics, publish)         16
0          (devote, examination, art)         15
5   (aware, historically, connection)         15
13                  (new, york, city)         12
38                 (queed, ob, spade)         11
8         (politics, publish, winter)         10
20    (guideline, contributor, issue)          9
35    (publish, collective, feminist)          9


In [30]:
print(df_bigrams)
pd.set_option('display.max_rows', None)
df_bigrams


                         bigram  frequency
32               (black, woman)         74
57                  (new, york)         73
9               (feminist, art)         44
21                (would, like)         34
2               (art, politics)         32
59                (woman, work)         30
54                 (woman, not)         28
33                  (year, ago)         28
256                 (year, old)         26
229            (work, together)         22
211                   (no, one)         21
238                (woman, art)         21
56               (woman, music)         20
42            (woman, movement)         20
69                   (one, day)         20
217               (many, woman)         20
6                (woman, aware)         19
182               (first, time)         19
68               (woman, woman)         19
26                (woman, make)         19
0         (devote, examination)         18
7         (aware, historically)         18
4        (c

,bigram,frequency
32,"(black, woman)",74
57,"(new, york)",73
9,"(feminist, art)",44
21,"(would, like)",34
2,"(art, politics)",32
59,"(woman, work)",30
54,"(woman, not)",28
33,"(year, ago)",28
256,"(year, old)",26
229,"(work, together)",22


I would like to filter out the following Bigrams:
(new, york)
(feminist, art)
(tell, u)
(united, state)
(york, city)
(queed, ob)
(puerto, rican)
(ob, spade)
(issue, typeset)
(gon, na)
(let, u)
(c.´s, mother)
(no, no)
(san, francisco)
(mother, m.)
(dear, consuelo)
(anais, nin)
(ruth, crawford)
(mother, k.)
(r., mother)
(vicki, yeah)
(myrna, zimmerman)
(1974, dear)
(dear, martha)
(crawford, seeger)
(12, sexuality)
(norma, jean)
(k., r.)
(k., mother)
(mother, j.)
(u, work)

In [31]:
print(df_trigrams)

                                  trigram  frequency
6               (feminist, art, politics)         19
4            (woman, aware, historically)         18
7                (art, politics, publish)         16
0              (devote, examination, art)         15
5       (aware, historically, connection)         15
13                      (new, york, city)         12
38                     (queed, ob, spade)         11
8             (politics, publish, winter)         10
20        (guideline, contributor, issue)          9
35        (publish, collective, feminist)          9
32                     (c.´s, mother, m.)          9
30              (woman, traditional, art)          9
2       (structure, collective, feminist)          9
14                      (live, new, york)          9
22                  (third, world, woman)          9
39                       (der, queed, ob)          7
1            (examination, art, politics)          7
41                  (click, click, click)     

I would like to filter out the following Trigrams: 
(feminist, art, politics)
(new, york, city)
(queed, ob, spade) 
(c.´s, mother, m.)
(der, queed, ob)
(click, click, click)
(consuelo, letter, no)
(dear, consuelo, letter)
(k., mother, k.)
(r., mother, j.)

In [32]:
EXCLUDE_BIGRAMS = [
    ("new", "york"),
    ("feminist", "art"),
    ("tell", "u"),
    ("united", "state"),
    ("york", "city"),
    ("queed", "ob"),
    ("puerto", "rican"),
    ("ob", "spade"),
    ("issue", "typeset"),
    ("gon", "na"),
    ("let", "u"),
    ("c.´s", "mother"),
    ("no", "no"),
    ("san", "francisco"),
    ("mother", "m."),
    ("dear", "consuelo"),
    ("anais", "nin"),
    ("ruth", "crawford"),
    ("mother", "k."),
    ("r.", "mother"),
    ("vicki", "yeah"),
    ("myrna", "zimmerman"),
    ("1974", "dear"),
    ("dear", "martha"),
    ("crawford", "seeger"),
    ("12", "sexuality"),
    ("norma", "jean"),
    ("k.", "r."),
    ("k.", "mother"),
    ("mother", "j."),
    ("u", "work")
]

EXCLUDE_TRIGRAMS = [
    ("feminist", "art", "politics"),
    ("new", "york", "city"),
    ("queed", "ob", "spade"),
    ("c.´s", "mother", "m."),
    ("der", "queed", "ob"),
    ("click", "click", "click"),
    ("consuelo", "letter", "no"),
    ("dear", "consuelo", "letter"),
    ("k.", "mother", "k."),
    ("r.", "mother", "j."),
    ("art", "politics", "publish"),
    ("devote", "examination", "art"),
    ("aware", "historically", "connection"),
    ("politics", "publish", "winter"),
    ("guideline", "contributor", "issue"),
    ("publish", "collective", "feminist"),
    ("structure", "collective", "feminist"),
    ("examination", "art", "politics"),
    ("winter", "spring", "summer"),
    ("contributor", "issue", "specific"),
    ("guideline", "prospective", "contributor"),
]

In [33]:
def apply_exclusions(df, col, exclusions):
    return df[~df[col].isin(exclusions)].reset_index(drop=True)

df_bigrams  = apply_exclusions(df_bigrams,  "bigram",  EXCLUDE_BIGRAMS)
df_trigrams = apply_exclusions(df_trigrams, "trigram", EXCLUDE_TRIGRAMS)

print(df_bigrams.head(10))
print(df_trigrams.head(10))

             bigram  frequency
0    (black, woman)         74
1     (would, like)         34
2   (art, politics)         32
3     (woman, work)         30
4      (woman, not)         28
5       (year, ago)         28
6       (year, old)         26
7  (work, together)         22
8         (no, one)         21
9      (woman, art)         21
                         trigram  frequency
0   (woman, aware, historically)         18
1      (woman, traditional, art)          9
2              (live, new, york)          9
3          (third, world, woman)          9
4   (collective, feminist, also)          7
5      (publish, winter, spring)          7
6  (indexed, alternative, press)          7
7   (alternative, press, centre)          6
8        (13, feminism, ecology)          6
9         (make, possible, part)          6


### 7. Saving the Results

In [34]:
df_bigrams.to_csv("/Users/sophiehamann/master-thesis-code/data/processed/10_h1_bigrams.csv", index=False)
df_trigrams.to_csv("/Users/sophiehamann/master-thesis-code/data/processed/10_h1_trigrams.csv", index=False)

print("done")
print(df_bigrams.head(10))
print(df_trigrams.head(10))

done
             bigram  frequency
0    (black, woman)         74
1     (would, like)         34
2   (art, politics)         32
3     (woman, work)         30
4      (woman, not)         28
5       (year, ago)         28
6       (year, old)         26
7  (work, together)         22
8         (no, one)         21
9      (woman, art)         21
                         trigram  frequency
0   (woman, aware, historically)         18
1      (woman, traditional, art)          9
2              (live, new, york)          9
3          (third, world, woman)          9
4   (collective, feminist, also)          7
5      (publish, winter, spring)          7
6  (indexed, alternative, press)          7
7   (alternative, press, centre)          6
8        (13, feminism, ecology)          6
9         (make, possible, part)          6
